<center><img src=https://www.mihaileric.com/static/model-selection-meme-bd4a6a86f615583d1a1bbc497ca4640e-67414.jpeg></center>
<center>img source: towards data science</center>

# <center><b>(nested) Cross-Validation⚙️</b></center>

**What you can expect from this notebook:** This notebook covers cross-validation in theory and practice and emphasises why nested cross-validation can and often should be used instead of simple cross-validation. 

**libraries utilised:**

In [1]:
#  for implementation
import numpy as np  # linear algebra
from copy import copy  # deep copies of objects

#  for evaluation
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

#  for data handeling and scaling
import pandas as pd  # data handeling
from sklearn.preprocessing import StandardScaler  # Standartise data

****
# <b>1 <span style="color:#ebd1a4">|</span> Basic cross-validation intuition</b>

**Main idea:** utilising ***all*** data avaliable for training ***and*** testing.

**Use cases:**
* Model evaluation (how does my model perform taking into account *all* my data?)
* Model selection (which model performs best taking into account *all* my data?)
* Hyperparameter tuning (Gridsearch -> like model selection but between the same model, just with different hyperparameter settings)

<center><img src=https://www.dummies.com/wp-content/uploads/9781119245513-fg1104.jpg></center>
<center>img source: ww.dumies.com</center>

**How it works:**
* **Partition** data into groups
* In a **rotating** manner:
    * **Set aside** a group for testing
    * Train models on the **rest**
    * Test these models with the testing group and keep track of the scores
    * Repeat but with **another group** being set aside as testing group
* After each group was used for testing:
    * **Average scores** to obtain an estimate **based on the whole data**
    
**Advantages:**
* The obtained scores are a better estimate of how the model will perform on real-world data than the ones one could obtain by just testing on a fixed subset of the data

**Disadvantages:**
* Each model has to be trained multiple times 

**Some common cross-validation variants:**
* ***K-Fold Cross-Validation:*** The most widely used one and also the one focussed on in this notebook:
    * Partition data into k folds(groups) of equal size
    * Test on one, train on the rest
    * Then rotate
    * **visualisation above**
* ***Leave one out Cross-Validation:***
    * Similar to K-Fold Cross-Validation
    * But partitions into n folds equal to the number of observations in the dataset
    * resulting in single item testing sets
* ***Stratfield Cross-Validation:***
    * Mainly used for classification problems
    * Similar to K-Fold Cross-Validation, it tests on a set percentage of the data
    * But partitions the data in a way that the mean response value is roughly equal across folds
    
<br>

#### **Python implementation of K-Fold Cross-Validation:**
This implementation will be used in an example after the next section.

In [2]:
#  sample implementation
def k_fold_crossval(models, X, y, k):
    #  initialise empty lists to store scores and trained models
    error_scores = []
    trained_models = []
    #  split dataset into folds
    folds_X = np.array_split(X, k)
    folds_y = np.array_split(y, k)
    #  iterate over folds
    for test_fold_i in range(k):
        #  log
        print(f"fold {test_fold_i+1}: ---------------------------------------------")
        #  specify train/test data
        X_train = folds_X[:test_fold_i] + folds_X[test_fold_i+1:]
        y_train = folds_y[:test_fold_i] + folds_y[test_fold_i+1:]
        X_test = folds_X[test_fold_i]
        y_test = folds_y[test_fold_i]
        #  train on everything but the test_fold
        fold_scores = []
        fold_models = []
        for model in self.models:
            #  make deep copy of model
            model = copy(model)
            #  fit and test
            model.fit(np.concatenate(X_train), np.concatenate(y_train))
            fold_models.append(model)
            score = model.score(X_test, y_test)
            fold_scores.append(score)
            #  log
            print(f"{model} trained, score = {score}")
        #  save fold scores and models
        error_scores.append(np.array(fold_scores))
        trained_models.append(np.array(fold_models, dtype=object))
    return error_scores, trained_models

****

# <b>2 <span style="color:#ebd1a4">|</span> The often accuring problem</b>

**Cross-validation can be used for both model evaluation and selection/hyperparameter tuning, but not for both at the same time without running into potential problems.**<br>

**Example Situation:**
* Someone performs 5-fold cross-validation to select between a catboost, xgboost and lightgbm algorithm based on which one obtains the lowest loss/highest score
* Since this already resulted in a loss value he reports that loss as an estimate of how the model is going to perform in production
* Unfortunately the model ends up performing slightly worse than expected

**What happened?**
* When selecting the best model, **bias** was introduced, since with that he did also choose the lowest loss as an estimate
* The obtained loss values are just estimates and have a sampling distribution
* When choosing the lowest value, it increases the chance of it coming from a lower end of that distribution 
* -> **After selecting a model based on loss, that loss estimate is biased**

****

# <b>3 <span style="color:#ebd1a4">|</span> How it can be fixed with nested cross-validation</b>

**The solution is stunningly simple:** The evaluation process is separated from the selection process, splitting into train, validation and test set, where the validation set is used for the model selection/hyperparameter optimization and the test set for the final evaluation.

**This can be done using all of the data for training, validation and testing, by utilising 2 cross-validation loops, nested into each other:**

<center><img src=https://vitalflux.com/wp-content/uploads/2020/08/Screenshot-2020-08-30-at-6.33.47-PM.png></center>
<center>img source: vitalflux.com</center>

**Step for step:**
* partition training data into folds (outer loop), **rotating testing fold**:
    * set aside testing set
    * partition the training folds in each iteration into folds (inner loop), **rotating the validation fold**:
        * set aside validation set
        * fit models on rest of training set
        * **evaluate on validation set**
        * repeat till all the training data from the outer loop was used for validation
    * **choose best model** based on scores of inner loop
    * **train** that model **on the whole training set** from the outer loop
    * evaluate on testing sett
    * repeat till all the data was used for testing
    
* This results in one(or more) best model(s) with an unbiased loss estimate
    
<br>

#### **Python implementation of K-Fold Cross-Validation:**
Taking the single loop created in the last section, it is extended in a way that it can be nested into each other. This is done by making it possible to be treated similar to a sklearn model(with .fit and .score):

In [3]:
#  turn into class to work with sklearn model pipeline and allow for easy nesting of cross-validation loops
class KfoldCrossval:
    def __init__(self, k, models, name="k_fold_crossval"):
        self.k = k
        self.models = models
        self.name = name
        
    def evaluate(self, X, y):
        self.fit(X, y)
        return self.error_scores, self.trained_models
        
    def fit(self, X, y, verbose=True):
        #  initialise empty lists to store scores and trained models
        self.error_scores = []
        self.trained_models = []
        #self.fold_scores = []
        #  split dataset into folds
        folds_X = np.array_split(X, self.k)
        folds_y = np.array_split(y, self.k)
        #  iterate over folds
        for test_fold_i in range(self.k):
            #  log
            if verbose:
                print(f"{self.name} fold {test_fold_i+1}: ---------------------------------------------")
            #  specify train/test data
            X_train = folds_X[:test_fold_i] + folds_X[test_fold_i+1:]
            y_train = folds_y[:test_fold_i] + folds_y[test_fold_i+1:]
            X_test = folds_X[test_fold_i]
            y_test = folds_y[test_fold_i]
            #  train on everything but the test_fold
            fold_scores = []
            fold_models = []
            for model in self.models:
                #  make deep copy of model
                model = copy(model)
                #  fit and test
                model.fit(np.concatenate(X_train), np.concatenate(y_train))
                fold_models.append(model)
                score = model.score(X_test, y_test)
                fold_scores.append(score)
                #  log
                if verbose:
                    print(f"{model} trained, score = {score}")
            #  save fold scores and models
            self.error_scores.append(np.array(fold_scores))
            self.trained_models.append(np.array(fold_models, dtype=object))
        
        #  just neccessary if inner loop:
        #  find best model
        scores = np.stack(self.error_scores).mean(axis=0)
        self.best_model = copy(np.array(self.models)[scores == scores.max()][0])
        #  fit best model on whole training data
        self.best_model.fit(X, y)
    
    def score(self, X_test, y_test):
        return self.best_model.score(X_test, y_test)

```
models = [model1, model2, ..., modeln]

#  example usage
inner = KfoldCrossval(10, models)
outer = KfoldCrossval(10, [inner])

outer.evaluate(X, y)
```

****

# <b>4 <span style="color:#ebd1a4">|</span> Sample usage of nested cross-validation</b>

In [4]:
house_data = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
scaler = StandardScaler()

# encode categorical features, fill nan values
for feature in house_data.columns:
    if house_data[feature].dtype == "object":
        #  categorical encoding: turns [a, b, b, c] into [1, 2, 2, 3]
        house_data[feature] = house_data[feature].astype("category").cat.codes 
        if house_data[feature].isna().sum() != 0:
            #  impute missing values with the mode of the corresponding variable
            house_data[feature].fillna(house_data[feature].mode(), inplace=True)
    else:
        if house_data[feature].isna().sum() != 0:
            #  impute missing values with the mean of the corresponding variable
            house_data[feature].fillna(house_data[feature].mean(), inplace=True)
            
features = house_data.loc[:, house_data.columns!="SalePrice"].to_numpy()
labels = house_data.loc[:, "SalePrice"].to_numpy()

scaler.fit(features)
features = scaler.transform(features)

In [5]:
house_data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,3,65.0,8450,1,-1,3,3,0,...,0,-1,-1,-1,0,2,2008,8,4,208500
1,2,20,3,80.0,9600,1,-1,3,3,0,...,0,-1,-1,-1,0,5,2007,8,4,181500
2,3,60,3,68.0,11250,1,-1,0,3,0,...,0,-1,-1,-1,0,9,2008,8,4,223500
3,4,70,3,60.0,9550,1,-1,0,3,0,...,0,-1,-1,-1,0,2,2006,8,0,140000
4,5,60,3,84.0,14260,1,-1,0,3,0,...,0,-1,-1,-1,0,12,2008,8,4,250000


In [6]:
#  extreamly basic configuration, just for demonstation purposes
models = [
    Ridge(1),
    DecisionTreeRegressor(max_depth=50, min_samples_split=10),
    RandomForestRegressor(n_estimators=50),
    #GradientBoostingRegressor(loss="squared_error", min_samples_split=10)
]

**6-fold inner, 6-fold outer cross-validation:**

In [7]:
inner = KfoldCrossval(6, models, "inner")
outer = KfoldCrossval(6, [inner], "outer")

best_scores, best_models = outer.evaluate(features, labels)

outer fold 1: ---------------------------------------------
inner fold 1: ---------------------------------------------
Ridge(alpha=1) trained, score = 0.8935371241927869
DecisionTreeRegressor(max_depth=50, min_samples_split=10) trained, score = 0.6719647890727651
RandomForestRegressor(n_estimators=50) trained, score = 0.9004278452416733
inner fold 2: ---------------------------------------------
Ridge(alpha=1) trained, score = 0.7816329876382804
DecisionTreeRegressor(max_depth=50, min_samples_split=10) trained, score = 0.8108930558553396
RandomForestRegressor(n_estimators=50) trained, score = 0.7809820689956996
inner fold 3: ---------------------------------------------
Ridge(alpha=1) trained, score = 0.8294021572143223
DecisionTreeRegressor(max_depth=50, min_samples_split=10) trained, score = 0.7820176702308266
RandomForestRegressor(n_estimators=50) trained, score = 0.8688492312705731
inner fold 4: ---------------------------------------------
Ridge(alpha=1) trained, score = 0.854984

In [8]:
best_models = [crossval[0].best_model for crossval in best_models]
print(best_models, best_scores)

[RandomForestRegressor(n_estimators=50), RandomForestRegressor(n_estimators=50), RandomForestRegressor(n_estimators=50), RandomForestRegressor(n_estimators=50), RandomForestRegressor(n_estimators=50), RandomForestRegressor(n_estimators=50)] [array([0.87279678]), array([0.89900148]), array([0.81719041]), array([0.87457542]), array([0.85477276]), array([0.79543448])]


As expectable the random forest regressor showed to be the best model, the mean of the outer crossvalidation score gives an estimate on how it will perform on real world data:

In [9]:
#  get average score -> better estimate
np.stack(best_scores).mean()

0.8522952223832578

**That's all for this notebook, have a great day and happy learning!👋**